# Construct a text measure with propaxis

This notebook walks through proposition writing, interactive editing, axis construction, scoring, and saving an instrument. The example measures support for transit funding. It illustrates the workflow; it is not a validated research instrument.

Install propaxis with the `models` and `notebook` extras in the notebook kernel environment. The first model call downloads E5-small. All subsequent inference is local.

In [ ]:
from IPython.display import display

from propaxis import Component, Construct, Encoder, Instrument
from propaxis.notebook import construct_editor

## 1. Define the component

Write complete claims at both poles. The positive pole expresses the target; the contrast pole expresses the relevant alternative. Comparable topics and grammar reduce unintended differences.

In [ ]:
component = Component(
    "transit_support",
    positive=[
        "The city should invest more in public transportation.",
        "Expanding bus and rail services should be a budget priority.",
        "Affordable public transit deserves public funding.",
    ],
    contrast=[
        "The city should reduce spending on public transportation.",
        "Expanding bus and rail services is not a budget priority.",
        "Public transit should receive less public funding.",
    ],
)
construct = Construct("transit_stance", [component], window=1, top_k=1)

## 2. Edit interactively

Use one proposition per line. Add a component if the definition requires another dimension. Click **Validate and preview** to check the specification. The form does not load a model or establish substantive validity.

In [ ]:
editor = construct_editor(construct)
display(editor)

Run the next cell after editing. It reads the current form values.

In [ ]:
construct = editor.build()
construct.to_dict()

## 3. Fit the axes and scaling parameters

Use representative reference texts. This small teaching sample includes supportive, opposing, and procedural statements. Use a larger dataset and sensitivity checks for research. Fitting constructs the axes and estimates scaling parameters without changing encoder weights.

In [ ]:
reference_texts = [
    "We should add bus routes to underserved neighborhoods.",
    "I support a larger budget for trains and buses.",
    "Affordable transit helps people get to work.",
    "The council should fund more frequent bus services.",
    "Our transport budget should focus on maintaining roads.",
    "We cannot afford another public rail expansion.",
    "Bus subsidies should be reduced next year.",
    "Public transportation already receives too much funding.",
    "The council will meet on Tuesday to discuss the budget.",
    "The transport committee published its annual report.",
    "Residents disagree about the planned bus route.",
    "The city is reviewing several transport proposals.",
]
encoder = Encoder("e5-small-v2")
instrument = construct.fit(reference_texts, encoder=encoder)

## 4. Inspect the axis

Anchor projections show where each proposition lies. The leave-one-out cosine measures how much the direction changes after removing a proposition. These are construction diagnostics, not validation against independent coding.

In [ ]:
instrument.diagnostics

## 5. Score and inspect new texts

Scores use the saved reference scaling parameters. A high score is stronger expression relative to that reference dataset. It is not a class probability.

In [ ]:
test_texts = [
    "We need frequent buses and the city should pay for them.",
    "The city should stop subsidizing public transportation.",
    "The council has scheduled a transport committee meeting.",
]
result = instrument.score(test_texts, ids=["support", "oppose", "procedural"])
result.to_pandas()

In [ ]:
result.evidence[0]

## 6. Save and reload

The instrument file contains the propositions, axes, scaling parameters, and model revisions. It excludes raw reference texts and model weights. Loading offline requires the recorded models in the local Hugging Face cache.

In [ ]:
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)
construct.save("outputs/transit_construct.json")
instrument.save("outputs/transit_instrument.json")
restored = Instrument.load("outputs/transit_instrument.json", local_files_only=True)
restored.score(test_texts).to_pandas()

## Next steps

For label prediction, use a separate calibration sample with `calibrate()` and evaluate on held-out texts. Keep repeated speakers or authors in the same partition. To add an entailment check, refit with `claim_check=Entailment()`; this downloads BART and requires more memory and computation.

Editing the propositions creates a new measure. Refit its scaling parameters and estimate new thresholds. Do not select changes using the final test set. See the README and the construct-design guide for examples.